# Spacepresso v16 — Multi-classe · 10 run · Ensemble normalizzato (come ensemble.py)

### Novità rispetto a v15
- **Ensemble normalizzato**: ogni modello viene normalizzato con P_LO/P_HI sulle good images
  prima di mediare — identico alla strategia di `ensemble.py`
- **Soglia pixel-level**: calcolata come media(p05_ano_pixel, p95_good_pixel) dalla
  distribuzione di tutti i pixel (non image-level)
- **Metriche su tutti i pixel delle true anomalies** + N hard good non in training
- **Riepilogo training** dopo il loop: tabella top-K per ogni classe
- **Riepilogo valutazione** dopo il loop: tabella metriche raw vs sogliato
- **Salvataggio modelli top-K** in `runs/timestamp/seg_heads/`

## 1. Setup

In [18]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

CUDA: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [19]:
!pip install -q timm

## 2. Mount

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
from pathlib import Path

DATA_ROOT  = Path('/content/drive/MyDrive/ConfusionModelsADL/dataset')
OUTPUT_DIR = Path('/content/drive/MyDrive/ConfusionModelsADL/output')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Dataset: {DATA_ROOT}")
print(f"Output:  {OUTPUT_DIR}")
print(f"Classi:  {[d.name for d in sorted(DATA_ROOT.iterdir()) if d.is_dir()]}")

Dataset: /content/drive/MyDrive/ConfusionModelsADL/dataset
Output:  /content/drive/MyDrive/ConfusionModelsADL/output
Classi:  ['class_01', 'class_02', 'class_03', 'class_04', 'class_05', 'class_06', 'class_07', 'class_08']


## 3. Configurazione

In [22]:
import random, numpy as np, torch

SEED              = 42
IMG_SIZE          = 224
PATCH_GRID        = 16
FEATURE_DIM       = 384
LAYERS_TO_USE     = [5, 8, 11]
MULTILAYER_DIM    = FEATURE_DIM * len(LAYERS_TO_USE)

TWERSKY_ALPHA     = 0.5
EPOCHS            = 500
PATIENCE          = 5
SAMPLES_PER_EPOCH = 200
BATCH_SIZE        = 64
LR                = 1e-3
BLUR_SIGMA        = 2

N_RUNS = 10   # training per classe
TOP_K  = 2    # modelli migliori da tenere

# ── Normalizzazione ensemble (come ensemble.py) ───────────────────────────────
P_LO = 0.5      # percentile basso per normalizzazione
P_HI = 99.999   # percentile alto per normalizzazione

# ── Parametri per classe ──────────────────────────────────────────────────────
CLASS_CONFIG = {
    'class_01': {'l1': 0.0,  'p_good': 0.40, 'p_real': 0.30},
    'class_02': {'l1': 0.0,  'p_good': 0.40, 'p_real': 0.30},
    'class_03': {'l1': 0.2,  'p_good': 0.40, 'p_real': 0.10},
    'class_04': {'l1': 0.2,  'p_good': 0.40, 'p_real': 0.30},
    'class_05': {'l1': 0.1,  'p_good': 0.40, 'p_real': 0.30},
    'class_06': {'l1': 0.0,  'p_good': 0.40, 'p_real': 0.30},
    'class_07': {'l1': 0.0,  'p_good': 0.40, 'p_real': 0.30},
    'class_08': {'l1': 0.0,  'p_good': 0.40, 'p_real': 0.30},
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device} | N_RUNS={N_RUNS} | TOP_K={TOP_K}")
print(f"P_LO={P_LO} | P_HI={P_HI}  (normalizzazione ensemble)")
print("\nParametri per classe:")
for cls, cfg in CLASS_CONFIG.items():
    cp = 1 - cfg['p_good'] - cfg['p_real']
    print(f"  {cls}: L1={cfg['l1']} | {cfg['p_good']*100:.0f}%g/{cfg['p_real']*100:.0f}%r/{cp*100:.0f}%cp")

def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

def worker_init_fn(worker_id):
    s=(torch.initial_seed()+worker_id)%2**32
    np.random.seed(s); random.seed(s)

set_all_seeds(SEED)
print("\nConfigurazione OK")

Device: cuda | N_RUNS=10 | TOP_K=2
P_LO=0.5 | P_HI=99.999  (normalizzazione ensemble)

Parametri per classe:
  class_01: L1=0.0 | 40%g/30%r/30%cp
  class_02: L1=0.0 | 40%g/30%r/30%cp
  class_03: L1=0.2 | 40%g/10%r/50%cp
  class_04: L1=0.2 | 40%g/30%r/30%cp
  class_05: L1=0.1 | 40%g/30%r/30%cp
  class_06: L1=0.0 | 40%g/30%r/30%cp
  class_07: L1=0.0 | 40%g/30%r/30%cp
  class_08: L1=0.0 | 40%g/30%r/30%cp

Configurazione OK


## 4. Moduli

### 4.1 Modello

In [23]:
import torch.nn as nn
import torch.nn.functional as F

class SegHead(nn.Module):
    def __init__(self, in_dim=MULTILAYER_DIM, hidden=128):
        super().__init__()
        self.conv1=nn.Conv2d(in_dim,hidden,1); self.bn1=nn.BatchNorm2d(hidden)
        self.conv2=nn.Conv2d(hidden,hidden,3,padding=1); self.bn2=nn.BatchNorm2d(hidden)
        self.conv3=nn.Conv2d(hidden,1,1)
    def forward(self,p):
        B=p.size(0)
        x=p.transpose(1,2).reshape(B,-1,PATCH_GRID,PATCH_GRID)
        x=F.relu(self.bn1(self.conv1(x)))
        x=F.relu(self.bn2(self.conv2(x)))
        x=self.conv3(x)
        return F.interpolate(x,size=(IMG_SIZE,IMG_SIZE),mode='bilinear',align_corners=False)

def tversky_loss(logits,target,alpha=TWERSKY_ALPHA,eps=1e-7):
    pred=torch.sigmoid(logits)
    pf=pred.view(pred.size(0),-1); tf=target.view(target.size(0),-1)
    tp=(pf*tf).sum(dim=1); fp=(pf*(1-tf)).sum(dim=1); fn=((1-pf)*tf).sum(dim=1)
    t=(tp+eps)/(tp+alpha*fp+(1-alpha)*fn+eps)
    return (1-t).mean()

def bce_tversky_l1_loss(logits, target, l1_lambda=0.0):
    bce=F.binary_cross_entropy_with_logits(logits,target)
    tv=tversky_loss(logits,target)
    base=bce+tv
    if l1_lambda==0.0: return base
    return base+l1_lambda*torch.sigmoid(logits).mean()

print("Modello OK")

Modello OK


### 4.2 Split + Cut-Paste

In [24]:
from PIL import Image
from scipy.ndimage import gaussian_filter
import gc
from collections import defaultdict

def get_object_mask(img, threshold=30):
    return (img.mean(axis=2) if img.ndim==3 else img) > threshold

def maybe_subcrop_large(crop_img, crop_mask, max_ratio=0.25):
    h,w=crop_mask.shape
    if (crop_mask>0).sum()/(h*w+1e-8)<max_ratio: return crop_img,crop_mask
    target_area=h*w*random.uniform(0.15,0.30)
    sub_h=max(8,min(int(np.sqrt(target_area*h/w)),h-2))
    sub_w=max(8,min(int(target_area/sub_h),w-2))
    ys,xs=np.where(crop_mask>0)
    if len(ys)==0: return crop_img,crop_mask
    cy,cx=random.choice(ys),random.choice(xs)
    y0=max(0,min(cy-sub_h//2,h-sub_h)); x0=max(0,min(cx-sub_w//2,w-sub_w))
    return (crop_img[y0:y0+sub_h,x0:x0+sub_w].copy(),
            crop_mask[y0:y0+sub_h,x0:x0+sub_w].copy())

def cut_paste(good_img, ano_img, ano_mask, scale_range=(0.4,1.2)):
    H,W=good_img.shape[:2]
    ys,xs=np.where(ano_mask>0)
    if len(ys)==0: return good_img.copy(),np.zeros((H,W),dtype=np.float32)
    y0,y1,x0,x1=ys.min(),ys.max()+1,xs.min(),xs.max()+1
    crop_img=ano_img[y0:y1,x0:x1].copy(); crop_mask=ano_mask[y0:y1,x0:x1].copy()
    crop_img,crop_mask=maybe_subcrop_large(crop_img,crop_mask)
    ch,cw=crop_img.shape[:2]
    if ch<4 or cw<4: return good_img.copy(),np.zeros((H,W),dtype=np.float32)
    scale=random.uniform(*scale_range)
    nh=max(6,min(int(ch*scale),H//2)); nw=max(6,min(int(cw*scale),W//2))
    crop_img=np.array(Image.fromarray(crop_img).resize((nw,nh),Image.BILINEAR))
    crop_mask=np.array(Image.fromarray(crop_mask).resize((nw,nh),Image.NEAREST))
    shift=np.random.randint(-15,16,size=3).reshape(1,1,3)
    crop_img=np.clip(crop_img.astype(np.int16)+shift,0,255).astype(np.uint8)
    obj_mask=get_object_mask(good_img); oys,oxs=np.where(obj_mask)
    if len(oys)==0:
        py=random.randint(0,H-nh); px=random.randint(0,W-nw)
    else:
        oy0,oy1=oys.min(),oys.max(); ox0,ox1=oxs.min(),oxs.max()
        py=random.randint(max(0,oy0-nh//4),max(0,min(H-nh,oy1-nh//2)))
        px=random.randint(max(0,ox0-nw//4),max(0,min(W-nw,ox1-nw//2)))
    synth=good_img.copy(); out_mask=np.zeros((H,W),dtype=np.float32)
    alpha=gaussian_filter((crop_mask>0).astype(np.float32),sigma=1.0)
    region=synth[py:py+nh,px:px+nw]
    synth[py:py+nh,px:px+nw]=(region*(1-alpha[:,:,None])+crop_img*alpha[:,:,None]).astype(np.uint8)
    out_mask[py:py+nh,px:px+nw]=alpha
    return synth,out_mask

def collect_sources_split(data_root, class_name, good_paths_all, seed=SEED):
    """
    Per ogni anomaly_type:
      1 view casuale -> val_items (raw) + cutpaste_sources
      restanti view  -> train_sources (real aug) + cutpaste_sources
    N_GOOD_VAL = n anomalie val (bilanciamento automatico).
    Hard good: selezionati casualmente, NON inclusi nel training.
    """
    train_sources=[]; cutpaste_sources=[]; val_ano=[]
    train_dir=data_root/class_name/'train'
    gt_dir=data_root/class_name/'ground_truth_train'
    if not (train_dir.is_dir() and gt_dir.is_dir()):
        return train_sources,cutpaste_sources,[]
    rng=random.Random(seed)
    for ano_gt_dir in sorted(gt_dir.iterdir()):
        ano_img_dir=train_dir/ano_gt_dir.name; pairs=[]
        for mask_path in sorted(ano_gt_dir.glob('*.png')):
            mask=np.array(Image.open(mask_path).convert('L'))
            if mask.max()==0: continue
            pairs.append((str(ano_img_dir/mask_path.name),mask))
        if not pairs: continue
        if len(pairs)==1:
            img_path,mask=pairs[0]; img=np.array(Image.open(img_path).convert('RGB'))
            entry={'image':img,'mask':mask,'anomaly_type':ano_gt_dir.name,'path':img_path}
            train_sources.append(entry); cutpaste_sources.append(entry); continue
        val_idx=rng.randint(0,len(pairs)-1)
        val_path,val_mask=pairs[val_idx]
        val_img=np.array(Image.open(val_path).convert('RGB'))
        val_ano.append({'path':val_path,'mask':val_mask,
                        'anomaly_type':ano_gt_dir.name,'is_good':False})
        cutpaste_sources.append({'image':val_img,'mask':val_mask,
                                  'anomaly_type':ano_gt_dir.name,'path':val_path})
        for i,(img_path,mask) in enumerate(pairs):
            if i==val_idx: continue
            img=np.array(Image.open(img_path).convert('RGB'))
            entry={'image':img,'mask':mask,'anomaly_type':ano_gt_dir.name,'path':img_path}
            train_sources.append(entry); cutpaste_sources.append(entry)
    # N_GOOD_VAL = numero anomalie val
    n_good_val=len(val_ano)
    rng_np=np.random.RandomState(seed)
    good_val_idx=rng_np.choice(len(good_paths_all),
                                size=min(n_good_val,len(good_paths_all)),replace=False)
    val_items=val_ano.copy()
    for idx in good_val_idx:
        val_items.append({'path':good_paths_all[idx],
                          'mask':np.zeros((1,1),dtype=np.uint8),
                          'anomaly_type':'good','is_good':True})
    by_type=defaultdict(int)
    for s in train_sources: by_type[s['anomaly_type']]+=1
    print(f"  {class_name}[{seed}]: train={len(train_sources)} cp_src={len(cutpaste_sources)} "
          f"val_ano={len(val_ano)} val_good={n_good_val}")
    return train_sources,cutpaste_sources,val_items

print("Split + cut-paste OK")

Split + cut-paste OK


### 4.3 Dataset

In [25]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

preprocess=transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225]),
])

class ImageFolderDataset(Dataset):
    def __init__(self,paths,tf): self.paths,self.tf=paths,tf
    def __len__(self): return len(self.paths)
    def __getitem__(self,i): return self.tf(Image.open(self.paths[i]).convert('RGB'))

class TestDataset(Dataset):
    def __init__(self,paths,tf): self.paths,self.tf=paths,tf
    def __len__(self): return len(self.paths)
    def __getitem__(self,i):
        path=self.paths[i]
        return self.tf(Image.open(path).convert('RGB')),Path(path).name

def _resize_mask(mask):
    if mask.shape==(IMG_SIZE,IMG_SIZE): return (mask>0).astype(np.float32)
    pil=Image.fromarray((mask>0).astype(np.uint8)*255)
    return (np.array(pil.resize((IMG_SIZE,IMG_SIZE),Image.NEAREST))>127).astype(np.float32)

class TrainSynthDataset(Dataset):
    def __init__(self,good_paths,train_sources,cutpaste_sources,n,tf,p_good,p_real):
        self.good_paths=good_paths; self.train_sources=train_sources
        self.cutpaste_sources=cutpaste_sources; self.n,self.tf=n,tf
        self.p_good,self.p_real=p_good,p_real
    def __len__(self): return self.n
    def __getitem__(self,i):
        r=random.random()
        if r<self.p_good or not self.train_sources:
            img=np.array(Image.open(random.choice(self.good_paths)).convert('RGB'))
            return self.tf(Image.fromarray(img)),torch.zeros(IMG_SIZE,IMG_SIZE)
        if r<self.p_good+self.p_real:
            src=random.choice(self.train_sources)
            return self.tf(Image.fromarray(src['image'])),torch.from_numpy(_resize_mask(src['mask'])).float()
        good_img=np.array(Image.open(random.choice(self.good_paths)).convert('RGB'))
        src=random.choice(self.cutpaste_sources)
        synth,mask=cut_paste(good_img,src['image'],src['mask'])
        return self.tf(Image.fromarray(synth)),torch.from_numpy(_resize_mask(mask.astype(np.uint8))).float()

print("Dataset OK")

Dataset OK


### 4.4 Feature extraction + Inferenza con 4-fold TTA

In [26]:
from tqdm import tqdm

_TTA_OPS = [
    ('orig',   lambda x: x,                         lambda x: x),
    ('flip_h', lambda x: torch.flip(x,[-1]),        lambda x: torch.flip(x,[-1])),
    ('flip_v', lambda x: torch.flip(x,[-2]),        lambda x: torch.flip(x,[-2])),
    ('rot90',  lambda x: torch.rot90(x,1,[-2,-1]),  lambda x: torch.rot90(x,-1,[-2,-1])),
]

@torch.no_grad()
def extract_multilayer_patches(model,x,layers=LAYERS_TO_USE):
    intermediates=model.get_intermediate_layers(x,n=layers,reshape=False,return_class_token=False,norm=True)
    return torch.cat(intermediates,dim=2)

@torch.no_grad()
def seghead_infer(model, paths, head, batch_size=32, seed=SEED, tta=True):
    g=torch.Generator(); g.manual_seed(seed)
    loader=DataLoader(TestDataset(paths,preprocess),batch_size=batch_size,
                      num_workers=4,pin_memory=True,generator=g,worker_init_fn=worker_init_fn)
    head.eval(); all_scores,fns=[],[]
    for imgs,names in tqdm(loader,desc='infer',leave=False):
        imgs=imgs.to(device,non_blocking=True); aug_scores=[]
        ops=_TTA_OPS if tta else [_TTA_OPS[0]]
        for name,fwd,inv in ops:
            x=fwd(imgs)
            patches=extract_multilayer_patches(model,x)
            s=torch.sigmoid(head(patches)).squeeze(1)
            s=inv(s); aug_scores.append(s.cpu().numpy()); del patches,s
        all_scores.append(np.stack(aug_scores).mean(0)); fns.extend(names)
        del imgs,aug_scores
    return np.concatenate(all_scores,axis=0),fns

print("Inferenza 4-fold TTA OK")

Inferenza 4-fold TTA OK


### 4.5 Checkpoint — salvataggio modelli top-K

In [27]:
import json
from datetime import datetime

def save_top_models(output_dir, class_models, config_dict, all_run_results):
    """
    Salva i TOP_K modelli per ogni classe in runs/timestamp/seg_heads/<class>_rank<k>.pt
    class_models: {class_name: [head1, head2, ...]}
    all_run_results: {class_name: [(val_ap, head), ...]} tutti i run ordinati
    """
    timestamp=datetime.now().strftime('%Y%m%d_%H%M%S')
    run_dir=output_dir/'runs'/timestamp
    (run_dir/'seg_heads').mkdir(parents=True,exist_ok=True)
    for class_name,heads in class_models.items():
        for rank,head in enumerate(heads):
            torch.save(head.state_dict(),
                       run_dir/'seg_heads'/f'{class_name}_rank{rank+1}.pt')
    with open(run_dir/'config.json','w') as f: json.dump(config_dict,f,indent=2)
    # Salva anche il riepilogo dei run
    summary={}
    for cls,results in all_run_results.items():
        summary[cls]=[{'rank':i+1,'val_ap':float(ap)} for i,(ap,_) in enumerate(results)]
    with open(run_dir/'run_summary.json','w') as f: json.dump(summary,f,indent=2)
    print(f'Modelli salvati -> {run_dir}')
    return run_dir

print("Checkpoint OK")

Checkpoint OK


### 4.6 Training

In [28]:
import copy
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm as tqdm_auto

@torch.no_grad()
def _val_ap(model, head, val_items, good_paths):
    head.eval(); all_preds,all_gt=[],[]
    for item in val_items:
        img=preprocess(Image.open(item['path']).convert('RGB')).unsqueeze(0).to(device)
        patches=extract_multilayer_patches(model,img)
        score=torch.sigmoid(head(patches)).squeeze().cpu().numpy()
        del img,patches
        if item.get('is_good',False):
            mask=np.zeros((IMG_SIZE,IMG_SIZE),dtype=np.uint8)
        else:
            mask=(item['mask']>0).astype(np.uint8)
            if mask.shape!=(IMG_SIZE,IMG_SIZE):
                mask=(np.array(Image.fromarray(mask*255).resize(
                    (IMG_SIZE,IMG_SIZE),Image.NEAREST))>127).astype(np.uint8)
        all_preds.append(score.flatten()); all_gt.append(mask.flatten())
    torch.cuda.empty_cache()
    gt=np.concatenate(all_gt)
    if gt.max()==0: return 0.0
    return float(average_precision_score(gt,np.concatenate(all_preds)))

def train_one(model, data_root, class_name, train_sources, cutpaste_sources,
              val_items, l1_lambda, p_good, p_real,
              epochs=EPOCHS, patience=PATIENCE, seed=SEED):
    good_paths=[str(p) for p in sorted((data_root/class_name/'train'/'good').glob('*.png'))]
    g=torch.Generator(); g.manual_seed(seed)
    loader=DataLoader(
        TrainSynthDataset(good_paths,train_sources,cutpaste_sources,
                          SAMPLES_PER_EPOCH,preprocess,p_good,p_real),
        batch_size=BATCH_SIZE,num_workers=4,pin_memory=True,
        generator=g,worker_init_fn=worker_init_fn)
    head=SegHead().to(device)
    optim=torch.optim.AdamW(head.parameters(),lr=LR,weight_decay=1e-4)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optim,T_max=epochs,eta_min=LR*0.01)
    best_ap=-1.0; best_state=None; no_improve=0; stopped_at=epochs

    # Checkpoint su Drive ad ogni miglioramento
    ckpt_dir=OUTPUT_DIR/'checkpoints'/class_name
    ckpt_dir.mkdir(parents=True,exist_ok=True)

    for epoch in range(epochs):
        head.train(); epoch_loss=0.0
        for imgs,masks in loader:
            imgs=imgs.to(device,non_blocking=True); masks=masks.to(device,non_blocking=True).unsqueeze(1)
            with torch.no_grad(): patches=extract_multilayer_patches(model,imgs)
            loss=bce_tversky_l1_loss(head(patches),masks,l1_lambda=l1_lambda)
            optim.zero_grad(); loss.backward(); optim.step(); scheduler.step()
            epoch_loss+=loss.item(); del imgs,masks,patches,loss
        avg_loss=epoch_loss/len(loader)
        val_ap=_val_ap(model,head,val_items,good_paths) if val_items else 0.0
        head.train()
        improved=val_ap>best_ap
        if improved:
            best_ap=val_ap; no_improve=0; del best_state; best_state=copy.deepcopy(head.state_dict())
            torch.save({'state_dict':head.state_dict(),'val_ap':best_ap,
                        'epoch':epoch,'seed':seed},
                       ckpt_dir/f'best_seed{seed}.pt')
        else: no_improve+=1
        star='*' if improved else ' '
        bar='#'*int(val_ap*20)+'.'*(20-int(val_ap*20))
        warn=f' >{no_improve}/{patience}' if no_improve>0 else ''
        print(f"    ep {epoch+1:>4}  loss={avg_loss:.4f}  val_ap={val_ap:.4f} [{bar}]  best={best_ap:.4f} {star}{warn}")
        if no_improve>=patience:
            stopped_at=epoch+1; break
    if best_state: head.load_state_dict(best_state)
    head.eval()
    del best_state; gc.collect(); torch.cuda.empty_cache()
    print(f"    -> best_val_ap={best_ap:.4f} (ep.{stopped_at})")
    return head, best_ap

print("Training OK")

Training OK


### 4.7 Ensemble normalizzato (come ensemble.py)

Per ogni modello:
1. Inferenza sulle good images → calcola `lo=P_LO` e `hi=P_HI` percentile
2. Normalizza lo score: `norm = clip((score - lo) / (hi - lo), 0, 1)`
3. Media degli score normalizzati tra i TOP_K modelli

Questo garantisce che modelli con scale di output diverse contribuiscano equamente.

In [29]:
@torch.no_grad()
def ensemble_infer(model, paths, heads, good_paths, tta=True,
                   p_lo=P_LO, p_hi=P_HI):
    """
    Inferenza ensemble normalizzata (come ensemble.py):
    - Per ogni head: inferenza su good_paths -> calcola lo/hi
    - Normalizza score con lo/hi -> media degli score normalizzati
    - 4-fold TTA incluso

    Args:
        paths:      immagini da inferire
        heads:      lista di SegHead (i TOP_K modelli)
        good_paths: good images per calcolare lo/hi di normalizzazione
    """
    normed_scores = []
    for rank, head in enumerate(heads):
        # Score sulle good per normalizzazione
        sh_good,_ = seghead_infer(model, good_paths, head, tta=False)
        lo = np.percentile(sh_good.flatten(), p_lo)
        hi = np.percentile(sh_good.flatten(), p_hi)

        # Score sulle immagini richieste con TTA
        sh,fns = seghead_infer(model, paths, head, tta=tta)

        # Blur
        if BLUR_SIGMA > 0:
            sh = np.stack([gaussian_filter(s, sigma=BLUR_SIGMA) for s in sh])

        # Normalizza
        norm = np.clip((sh - lo) / (hi - lo + 1e-8), 0, 1).astype(np.float32)
        normed_scores.append(norm)
        print(f"    modello {rank+1}/{len(heads)}: lo={lo:.4f} hi={hi:.4f}")

    # Media degli score normalizzati
    ensemble = np.stack(normed_scores).mean(0)  # (N, H, W)
    return ensemble, fns

print("Ensemble normalizzato OK")

Ensemble normalizzato OK


### 4.8 Valutazione estesa

In [30]:
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.metrics import average_precision_score,roc_auc_score,roc_curve,precision_recall_curve

def _resize_mask_eval(mask):
    return (np.array(Image.fromarray(mask).resize((IMG_SIZE,IMG_SIZE),Image.NEAREST))>0).astype(np.uint8)

def _compute_metrics_pixel(gt_pixel_flat, scores_pixel_flat,
                            gt_image, scores_image):
    """
    Metriche calcolate su tutti i pixel delle true anomalies.
    gt_pixel_flat: tutti i pixel (good=0, anomaly=1)
    scores_pixel_flat: score per ogni pixel
    gt_image, scores_image: per le metriche image-level
    """
    pixel_auroc=float(roc_auc_score(gt_pixel_flat,scores_pixel_flat))
    pixel_ap=float(average_precision_score(gt_pixel_flat,scores_pixel_flat))
    fpr_arr,tpr_arr,_=roc_curve(gt_pixel_flat,scores_pixel_flat)
    idx95=np.searchsorted(tpr_arr,0.95)
    fpr_at_tpr95=float(fpr_arr[min(idx95,len(fpr_arr)-1)])
    prec_arr,rec_arr,_=precision_recall_curve(gt_pixel_flat,scores_pixel_flat)
    def pat(t):
        idx=np.where(rec_arr>=t)[0]; return float(prec_arr[idx[-1]]) if len(idx) else 0.0
    n_good=int((np.array(gt_image)==0).sum())
    good_max=scores_image[:n_good]; ano_max=scores_image[n_good:]
    p95_good_img=float(np.percentile(good_max,95)) if len(good_max)>0 else 0.0
    p05_ano_img =float(np.percentile(ano_max,5))   if len(ano_max)>0  else 0.0
    return {
        'pixel_auroc':pixel_auroc,'pixel_ap':pixel_ap,'fpr_at_tpr95':fpr_at_tpr95,
        'prec_at_rec50':pat(0.50),'prec_at_rec80':pat(0.80),
        'image_auroc':float(roc_auc_score(gt_image,scores_image)),
        'image_ap':float(average_precision_score(gt_image,scores_image)),
        'mean_good':float(good_max.mean()),'mean_ano':float(ano_max.mean()) if len(ano_max)>0 else 0.0,
        'score_gap':float(ano_max.mean()-good_max.mean()) if len(ano_max)>0 else 0.0,
        'p95_good':p95_good_img,'p05_ano':p05_ano_img,
        'separability':p05_ano_img-p95_good_img,
    }

def _print_comparison(m_raw, m_thr, threshold, class_name):
    keys=[('pixel_auroc','pixel_auroc'),('pixel_ap','pixel_ap <- principale'),
          ('fpr_at_tpr95','fpr@tpr95 <- noise'),('prec_at_rec50','prec@rec50'),
          ('prec_at_rec80','prec@rec80'),('image_auroc','image_auroc'),
          ('image_ap','image_ap'),('mean_good','mean_good'),
          ('score_gap','score_gap'),('separability','separability')]
    print(f"\n  {class_name}  threshold={threshold:.4f}")
    print(f"  {'_'*62}")
    print(f"  {'Metrica':<30} {'Raw':>8}  {'Sogliato':>10}  {'Delta':>7}")
    print(f"  {'_'*62}")
    for k,label in keys:
        vr=m_raw[k]; vt=m_thr[k]; delta=vt-vr
        sym='up' if delta>0.001 else ('dn' if delta<-0.001 else '==')
        print(f"  {label:<30} {vr:>8.3f}  {vt:>10.3f}  {sym} {abs(delta):>5.3f}")
    print(f"  {'_'*62}")

def evaluate_ensemble(model, data_root, class_name, heads, max_good=150, seed=0):
    """
    Valuta l'ensemble su TUTTE le anomalie reali.
    - Good per metriche: max_good good images NON in training
    - Threshold: media(p05_ano_pixel, p95_good_pixel) dalla distribuzione pixel
    - Metriche calcolate su tutti i pixel delle true anomalies
    """
    # Tutte le anomalie reali
    gt_dir   =data_root/class_name/'ground_truth_train'
    train_dir=data_root/class_name/'train'
    all_real=[]
    for ano_gt_dir in sorted(gt_dir.iterdir()):
        ano_img_dir=train_dir/ano_gt_dir.name
        for mask_path in sorted(ano_gt_dir.glob('*.png')):
            mask=np.array(Image.open(mask_path).convert('L'))
            if mask.max()==0: continue
            all_real.append({'path':str(ano_img_dir/mask_path.name),'mask':mask,
                             'ano_type':ano_gt_dir.name,'view':mask_path.stem})

    # Good images (tutte, poi subsampla per metriche)
    all_good_paths=[str(p) for p in sorted((data_root/class_name/'train'/'good').glob('*.png'))]
    rng=np.random.default_rng(seed)
    if max_good and len(all_good_paths)>max_good:
        idx=rng.choice(len(all_good_paths),size=max_good,replace=False)
        good_eval=[all_good_paths[i] for i in sorted(idx)]
    else:
        good_eval=all_good_paths
    n_good=len(good_eval)

    all_paths=good_eval+[r['path'] for r in all_real]

    # Ensemble normalizzato
    print(f"  Ensemble normalizzato ({len(heads)} modelli, 4-fold TTA)...")
    ens_scores,_=ensemble_infer(model,all_paths,heads,all_good_paths,tta=True)
    # ens_scores: (N, H, W) già normalizzato

    # Ground truth pixel-level
    zero=np.zeros((IMG_SIZE,IMG_SIZE),dtype=np.uint8)
    gt_masks_all=[zero]*n_good+[
        (np.array(Image.fromarray(r['mask']).resize((IMG_SIZE,IMG_SIZE),Image.NEAREST))>0).astype(np.uint8)
        for r in all_real]

    # ── Distribuzione pixel: tutti i pixel di TUTTE le anomalie ──────────────
    pixels_in=[]; pixels_out=[]
    for i,r in enumerate(all_real):
        score=ens_scores[n_good+i]
        mask_b=(np.array(Image.fromarray(r['mask']).resize((IMG_SIZE,IMG_SIZE),Image.NEAREST))>0)
        pixels_in.append(score[mask_b].flatten())
        pixels_out.append(score[~mask_b].flatten())
    pixels_in =np.concatenate(pixels_in)
    pixels_out=np.concatenate(pixels_out)

    # Threshold = media(p05_ano_pixel, p95_good_pixel)
    p95_good_pixel=float(np.percentile(pixels_out,95))
    p05_ano_pixel =float(np.percentile(pixels_in,5))
    threshold_pixel=(p95_good_pixel+p05_ano_pixel)/2.0
    print(f"  p95_good_pixel={p95_good_pixel:.4f} | p05_ano_pixel={p05_ano_pixel:.4f}")
    print(f"  threshold_pixel = ({p95_good_pixel:.4f}+{p05_ano_pixel:.4f})/2 = {threshold_pixel:.4f}")

    # ── Score sogliato (image-level: azzera mappe sotto soglia) ──────────────
    sh_thr=np.stack([s if s.max()>=threshold_pixel else np.zeros_like(s) for s in ens_scores])

    # ── Metriche pixel-level su tutti i pixel ────────────────────────────────
    gt_pixel_flat=np.stack(gt_masks_all).flatten().astype(int)
    if gt_pixel_flat.min()==gt_pixel_flat.max(): return {}
    im_sc_raw=ens_scores.reshape(len(all_paths),-1).max(axis=1)
    im_sc_thr=sh_thr.reshape(len(all_paths),-1).max(axis=1)
    im_lab=[0]*n_good+[1]*len(all_real)

    m_raw=_compute_metrics_pixel(gt_pixel_flat,ens_scores.flatten(),im_lab,im_sc_raw)
    m_thr=_compute_metrics_pixel(gt_pixel_flat,sh_thr.flatten(),im_lab,im_sc_thr)

    _print_comparison(m_raw,m_thr,threshold_pixel,class_name)

    # ── Plot 1: distribuzione pixel ────────────────────────────────────────────
    fig,axes=plt.subplots(1,2,figsize=(16,5))
    axes[0].hist(pixels_out,bins=100,alpha=0.6,color='steelblue',
                 label=f'Pixel normali n={len(pixels_out):,}',density=True)
    axes[0].hist(pixels_in, bins=100,alpha=0.6,color='crimson',
                 label=f'Pixel anomali n={len(pixels_in):,}',density=True)
    axes[0].axvline(p95_good_pixel,color='navy',ls='--',lw=1.5,label=f'p95 good={p95_good_pixel:.3f}')
    axes[0].axvline(p05_ano_pixel,color='darkred',ls='--',lw=1.5,label=f'p05 ano={p05_ano_pixel:.3f}')
    axes[0].axvline(threshold_pixel,color='green',ls='-',lw=2,label=f'threshold={threshold_pixel:.3f}')
    axes[0].set_xlabel('Score pixel'); axes[0].set_ylabel('Densita')
    axes[0].set_title(f'Distribuzione pixel — {class_name}')
    axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
    gm=im_sc_raw[:n_good]; am=im_sc_raw[n_good:]
    axes[1].hist(gm,bins=30,alpha=0.6,color='steelblue',label='Good')
    axes[1].hist(am,bins=30,alpha=0.6,color='crimson',label='Anomaly')
    axes[1].axvline(threshold_pixel,color='green',ls='-',lw=2,label=f'threshold={threshold_pixel:.3f}')
    axes[1].set_xlabel('Max pixel score'); axes[1].set_ylabel('Count')
    axes[1].set_title(f'Score distribution (image-level) — {class_name}')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    # ── Plot 2: ROC ────────────────────────────────────────────────────────────
    fpr_r,tpr_r,_=roc_curve(gt_pixel_flat,ens_scores.flatten())
    fpr_t,tpr_t,_=roc_curve(gt_pixel_flat,sh_thr.flatten())
    fig,ax=plt.subplots(figsize=(7,6))
    ax.plot(fpr_r,tpr_r,label=f'Raw AUC={m_raw["pixel_auroc"]:.3f}',color='steelblue',lw=2)
    ax.plot(fpr_t,tpr_t,label=f'Sogliato AUC={m_thr["pixel_auroc"]:.3f}',color='crimson',lw=2,ls='--')
    ax.plot([0,1],[0,1],'k--',lw=0.8)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'Pixel ROC — {class_name}'); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    # ── Plot 3: ogni anomalia reale ────────────────────────────────────────────
    print(f"\n  Heatmaps su tutte le anomalie reali ({len(all_real)}):")
    for i,r in enumerate(all_real):
        score=ens_scores[n_good+i]; score_thr=sh_thr[n_good+i]
        img_arr=np.array(Image.open(r['path']).convert('RGB').resize((IMG_SIZE,IMG_SIZE)))
        mask_arr=(np.array(Image.fromarray(r['mask']).resize(
            (IMG_SIZE,IMG_SIZE),Image.NEAREST))>0).astype(np.uint8)
        mx=score.max(); mx_t=score_thr.max()
        svis_r=np.clip(score/(mx+1e-8),0,1).astype(np.float32)
        svis_t=np.clip(score_thr/(mx_t+1e-8),0,1).astype(np.float32) if mx_t>1e-8 else np.zeros_like(score)
        col_title='AZZERATA' if mx_t<threshold_pixel else f'max={mx_t:.3f}'
        col_color='red' if mx_t<threshold_pixel else 'green'
        fig,axes=plt.subplots(1,5,figsize=(17.5,3.5))
        axes[0].imshow(img_arr); axes[0].set_title(f'{r["ano_type"]}\n{r["view"]}',fontsize=8)
        axes[1].imshow(mask_arr,cmap='gray',vmin=0,vmax=1); axes[1].set_title('GT Mask',fontsize=8)
        axes[2].imshow(svis_r,cmap='hot',vmin=0,vmax=1); axes[2].set_title(f'Raw max={mx:.3f}',fontsize=8)
        axes[3].imshow(svis_t,cmap='hot',vmin=0,vmax=1); axes[3].set_title(col_title,fontsize=8,color=col_color)
        axes[4].imshow(img_arr); axes[4].imshow(svis_t,cmap='hot',alpha=0.55,vmin=0,vmax=1)
        axes[4].set_title('Overlay sogliato',fontsize=8)
        for ax in axes: ax.axis('off')
        plt.tight_layout(); plt.show()

    m_thr['threshold_pixel']=threshold_pixel
    m_thr['p95_good_pixel']=p95_good_pixel
    m_thr['p05_ano_pixel']=p05_ano_pixel
    return {'raw':m_raw,'thresholded':m_thr,
            'threshold_pixel':threshold_pixel,
            'p95_good_pixel':p95_good_pixel,
            'p05_ano_pixel':p05_ano_pixel,
            'n_good':n_good,'n_anomaly':len(all_real)}

print("Valutazione OK")

Valutazione OK


### 4.9 Submission

In [31]:
import pandas as pd

def float_matrix_to_q8rle(x):
    q=np.clip(np.rint(np.asarray(x,dtype=np.float32)*255),0,255).astype(np.uint8)
    h,w=q.shape; flat=q.T.reshape(-1)
    if flat.size==0: return f'q8rle {h} {w}'
    cuts=np.flatnonzero(flat[1:]!=flat[:-1])+1
    starts=np.r_[0,cuts]; ends=np.r_[cuts,flat.size]
    parts=['q8rle',str(h),str(w)]
    for v,n in zip(flat[starts],ends-starts): parts+=[str(int(v)),str(int(n))]
    return ' '.join(parts)

def encode_and_merge(ensemble_scores, thresholds, run_dir, best_path=None):
    if best_path is None:
        best_path=Path('/content/drive/MyDrive/ConfusionModelsADL/submission_best.csv')
    sub_dir=run_dir/'submissions'; sub_dir.mkdir(exist_ok=True)
    rows_raw=[]; rows_thr=[]
    for class_name,res in ensemble_scores.items():
        thr=thresholds.get(class_name,0.0)
        for fn,score in zip(res['fns'],res['scores']):
            ens=np.clip(score,0,1).astype(np.float32)
            ens_thr=ens if ens.max()>=thr else np.zeros_like(ens)
            rows_raw.append({'ID':fn[:-4],'Label':float_matrix_to_q8rle(ens)})
            rows_thr.append({'ID':fn[:-4],'Label':float_matrix_to_q8rle(ens_thr)})
    def _save_merge(rows,suffix):
        csv=sub_dir/f'submission_{suffix}.csv'
        pd.DataFrame(rows).to_csv(csv,index=False)
        if best_path.exists():
            df_b=pd.read_csv(best_path); df_n=pd.read_csv(csv)
            lk=df_n.set_index('ID')['Label'].to_dict()
            mask=df_b['ID'].isin(lk)
            df_m=df_b.copy(); df_m.loc[mask,'Label']=df_m.loc[mask,'ID'].map(lk)
            mp=sub_dir/f'submission_merged_{suffix}.csv'
            df_m.to_csv(mp,index=False)
            print(f'  merged_{suffix} -> {mp}  ({mask.sum()} sostituite)')
            return mp
        return csv
    m_raw=_save_merge(rows_raw,'raw')
    m_thr=_save_merge(rows_thr,'thr')
    return m_raw,m_thr

print("Submission OK")

Submission OK


## 5. Training — 10 run per classe, tieni i 2 migliori

In [32]:
import warnings
warnings.filterwarnings('ignore',message='xFormers is not available')
set_all_seeds(SEED)
print(f"Device: {device} | N_RUNS={N_RUNS} | TOP_K={TOP_K}")

Device: cuda | N_RUNS=10 | TOP_K=2


In [33]:
print('Caricamento DINOv2...')
dinov2=torch.hub.load('facebookresearch/dinov2','dinov2_vits14',verbose=False)
dinov2=dinov2.to(device).eval()
for p in dinov2.parameters(): p.requires_grad=False
print('DINOv2 OK')

Caricamento DINOv2...
DINOv2 OK


In [ ]:
best_models = {}      # {class_name: [head1, head2]}
all_run_results = {}  # {class_name: [(val_ap, head), ...]} tutti i run ordinati

class_dirs=sorted([d for d in DATA_ROOT.iterdir()
                   if d.is_dir() and (d/'ground_truth_train').is_dir()])

for class_dir in class_dirs:
    class_name=class_dir.name
    cfg=CLASS_CONFIG.get(class_name,{'l1':0.0,'p_good':0.6,'p_real':0.2})
    l1,p_good,p_real=cfg['l1'],cfg['p_good'],cfg['p_real']
    good_paths_all=[str(p) for p in sorted((DATA_ROOT/class_name/'train'/'good').glob('*.png'))]

    print(f"\n{'='*60}")
    print(f"  {class_name}  L1={l1} | {p_good*100:.0f}%g/{p_real*100:.0f}%r/{(1-p_good-p_real)*100:.0f}%cp")
    print(f"{'='*60}")

    run_results=[]
    for run_idx in range(N_RUNS):
        run_seed=SEED+run_idx*100+hash(class_name)%1000
        print(f"\n  Run {run_idx+1}/{N_RUNS}  [seed={run_seed}]")
        train_src,cp_src,val_items=collect_sources_split(
            DATA_ROOT,class_name,good_paths_all,seed=run_seed)
        if not train_src and not cp_src:
            print(f"  {class_name}: nessuna anomalia, skip"); break
        head,best_ap=train_one(
            dinov2,DATA_ROOT,class_name,train_src,cp_src,val_items,
            l1_lambda=l1,p_good=p_good,p_real=p_real,
            epochs=EPOCHS,patience=PATIENCE,seed=run_seed)
        run_results.append((best_ap,head))

    run_results.sort(key=lambda x:x[0],reverse=True)
    all_run_results[class_name]=run_results
    best_models[class_name]=[h for _,h in run_results[:TOP_K]]

print("\nTraining completato!")


  class_01  L1=0.0 | 40%g/30%r/30%cp

  Run 1/10  [seed=69]
  class_01[69]: train=16 cp_src=20 val_ano=4 val_good=4
    ep    1  loss=1.5025  val_ap=0.4050 [########............]  best=0.4050 *
    ep    2  loss=1.3541  val_ap=0.8363 [################....]  best=0.8363 *
    ep    3  loss=1.2171  val_ap=0.8820 [#################...]  best=0.8820 *
    ep    4  loss=1.1393  val_ap=0.8809 [#################...]  best=0.8820   >1/5
    ep    5  loss=1.1200  val_ap=0.8798 [#################...]  best=0.8820   >2/5
    ep    6  loss=1.0673  val_ap=0.8902 [#################...]  best=0.8902 *
    ep    7  loss=0.9982  val_ap=0.8967 [#################...]  best=0.8967 *
    ep    8  loss=0.9724  val_ap=0.9030 [##################..]  best=0.9030 *
    ep    9  loss=0.9719  val_ap=0.8972 [#################...]  best=0.9030   >1/5
    ep   10  loss=0.9542  val_ap=0.9019 [##################..]  best=0.9030   >2/5
    ep   11  loss=0.9178  val_ap=0.9054 [##################..]  best=0.9054 *
    e

### Riepilogo training

In [ ]:
print("\n" + "="*65)
print("  RIEPILOGO TRAINING")
print("="*65)
print(f"  {'Classe':<12} {'Run':>4}  {'Val AP':>8}  {'Rank'}")
print("  " + "-"*50)
for class_name, results in sorted(all_run_results.items()):
    for rank,(ap,_) in enumerate(results):
        marker = f'  <- TOP {rank+1}' if rank < TOP_K else ''
        run_seed_disp = SEED + rank*100 + hash(class_name)%1000
        print(f"  {class_name:<12} s={run_seed_disp:<6}  {ap:>8.4f}  #{rank+1}{marker}")
    print()
print("="*65)
print(f"  Modelli selezionati per ensemble (TOP_{TOP_K}):")
for cls,heads in sorted(best_models.items()):
    aps=[ap for ap,_ in all_run_results[cls][:TOP_K]]
    print(f"  {cls}: " + " | ".join([f"#{i+1} ap={a:.4f}" for i,a in enumerate(aps)]))

## 6. Valutazione ensemble

In [ ]:
all_metrics={}
thresholds={}  # {class_name: threshold_pixel}

for class_name,heads in best_models.items():
    print(f"\n{'='*60}")
    print(f"  Valutazione: {class_name}  ({len(heads)} modelli)")
    print(f"{'='*60}")
    m=evaluate_ensemble(dinov2,DATA_ROOT,class_name,heads,max_good=150,seed=SEED)
    if m:
        all_metrics[class_name]=m
        thresholds[class_name]=m['threshold_pixel']
        torch.cuda.empty_cache()

### Riepilogo valutazione

In [ ]:
print("\n" + "="*80)
print("  RIEPILOGO VALUTAZIONE ENSEMBLE")
print("="*80)
print(f"  {'Classe':<12} {'PxAP_r':>7} {'PxAP_t':>7} {'fpr@95_r':>9} {'fpr@95_t':>9} "
      f"{'Gap_t':>7} {'Sep_t':>7} {'Thr':>7}")
print("  " + "-"*75)
for cls,m in sorted(all_metrics.items()):
    r=m['raw']; t=m['thresholded']; thr=m['threshold_pixel']
    print(f"  {cls:<12} {r['pixel_ap']:>7.3f} {t['pixel_ap']:>7.3f} "
          f"{r['fpr_at_tpr95']:>9.3f} {t['fpr_at_tpr95']:>9.3f} "
          f"{t['score_gap']:>7.3f} {t['separability']:>7.3f} {thr:>7.4f}")
print("  " + "-"*75)
if all_metrics:
    def mean_key(k,src):
        vals=[m[src][k] for m in all_metrics.values() if src in m and k in m[src]]
        return np.mean(vals) if vals else 0.0
    print(f"  {'MEAN':<12} {mean_key('pixel_ap','raw'):>7.3f} {mean_key('pixel_ap','thresholded'):>7.3f} "
          f"{mean_key('fpr_at_tpr95','raw'):>9.3f} {mean_key('fpr_at_tpr95','thresholded'):>9.3f} "
          f"{mean_key('score_gap','thresholded'):>7.3f} {mean_key('separability','thresholded'):>7.3f}")
print("="*80)

print("\nDettaglio metriche per classe (raw vs sogliato):")
for cls,m in sorted(all_metrics.items()):
    r=m['raw']; t=m['thresholded']; thr=m['threshold_pixel']
    keys=[('pixel_auroc','pixel_auroc'),('pixel_ap','pixel_ap <- principale'),
          ('fpr_at_tpr95','fpr@tpr95 <- noise'),('prec_at_rec50','prec@rec50'),
          ('prec_at_rec80','prec@rec80'),('image_auroc','image_auroc'),
          ('image_ap','image_ap'),('mean_good','mean_good'),
          ('score_gap','score_gap'),('separability','separability')]
    print(f"\n  {cls}  (threshold={thr:.4f}  p95_good_px={m['p95_good_pixel']:.4f}  p05_ano_px={m['p05_ano_pixel']:.4f})")
    print(f"  {'_'*62}")
    print(f"  {'Metrica':<30} {'Raw':>8}  {'Sogliato':>10}  {'Delta':>7}")
    print(f"  {'_'*62}")
    for k,label in keys:
        vr=r.get(k,0); vt=t.get(k,0); delta=vt-vr
        sym='up' if delta>0.001 else ('dn' if delta<-0.001 else '==')
        print(f"  {label:<30} {vr:>8.3f}  {vt:>10.3f}  {sym} {abs(delta):>5.3f}")
    print(f"  {'_'*62}")

## 7. Salvataggio modelli top-K

In [ ]:
config_dict={
    'seed':SEED,'n_runs':N_RUNS,'top_k':TOP_K,
    'epochs':EPOCHS,'patience':PATIENCE,
    'twersky_alpha':TWERSKY_ALPHA,'blur_sigma':BLUR_SIGMA,
    'p_lo':P_LO,'p_hi':P_HI,
    'class_config':{cls:cfg for cls,cfg in CLASS_CONFIG.items()},
    'data_root':str(DATA_ROOT),
}
run_dir=save_top_models(OUTPUT_DIR,best_models,config_dict,all_run_results)
print(f'\nModelli e config salvati in: {run_dir}')

## 8. Inferenza & Submission

In [ ]:
print('Inferenza ensemble normalizzata su test set...')
ensemble_scores={}

for class_name,heads in best_models.items():
    test=[str(p) for p in sorted((DATA_ROOT/class_name/'test').glob('*.png'))]
    good_paths_all=[str(p) for p in sorted(
        (DATA_ROOT/class_name/'train'/'good').glob('*.png'))]
    print(f"\n  {class_name}: {len(test)} test | {len(heads)} modelli | 4-fold TTA")
    ens,fns=ensemble_infer(dinov2,test,heads,good_paths_all,tta=True)
    ensemble_scores[class_name]={'fns':fns,'scores':ens}
    torch.cuda.empty_cache()

print('\nInferenza completata!')

In [ ]:
merged_raw,merged_thr=encode_and_merge(ensemble_scores,thresholds,run_dir)
print(f'\nSubmission raw:      {merged_raw}')
print(f'Submission sogliata: {merged_thr}')
print('\nSubmitta entrambe e confronta gli score!')